In [ ]:
!pip install -U sentence-transformers rank_bm25 pandas numpy
!pip install --upgrade bottleneck

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
import os
import re
import sqlite3
import pickle
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

DB_PATH = Path("/content/drive/MyDrive/MAI NLP WS26 27/RAG Documents/student_assistant_rag_0_6B.db")
BM25_CACHE = Path("/content/drive/MyDrive/MAI NLP WS26 27/bm25_chunks.pkl")
EMBEDDING_MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
TOP_K = 10
EMBED_TYPE = np.float32

def connect_db():
    # Enable foreign-key enforcement for this connection.
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    return conn

def tokenise(text):
    """Unicode-aware tokeniser preserving words and hyphenated terms."""
    return re.findall(r"\b[\w-]+\b", str(text).lower(), flags=re.UNICODE)


In [ ]:
def read_bm25_corpus():
    with connect_db() as conn:
        rows = conn.execute(
            """
            SELECT chunk_id, embed_text
            FROM chunks
            WHERE embed_text IS NOT NULL AND trim(embed_text) <> ''
            ORDER BY chunk_id
            """
        ).fetchall()

    chunk_ids = [int(row["chunk_id"]) for row in rows]
    texts = [row["embed_text"] for row in rows]
    return chunk_ids, texts


def get_db_signature():
    # Fast structural check. Does not detect edits to existing text with same IDs.
    with connect_db() as conn:
        row = conn.execute(
            "SELECT COUNT(*) AS n, COALESCE(MAX(chunk_id), 0) AS max_id FROM chunks"
        ).fetchone()
    return {"row_count": int(row["n"]), "max_chunk_id": int(row["max_id"])}


def build_and_save_bm25(cache_path=BM25_CACHE):
    chunk_ids, texts = read_bm25_corpus()
    if not texts:
        raise ValueError("No non-empty chunks.embed_text rows found.")

    tokenized = [tokenise(text) for text in texts]
    print(tokenized)
    bm25_index = BM25Okapi(tokenized)

    payload = {
        "format_version": 1,
        "built_at": datetime.now(timezone.utc).isoformat(),
        "db_path": str(DB_PATH.resolve()),
        "signature": get_db_signature(),
        "chunk_ids": chunk_ids,
        "bm25": bm25_index,
    }

    cache_path = Path(cache_path)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    with cache_path.open("wb") as f:
        pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)

    print(f"Saved BM25 index: {cache_path.resolve()}")
    print(f"Indexed chunks: {len(chunk_ids):,}")
    print(f"Built at: {payload['built_at']}")
    return payload

# Run once, then comment out or skip on normal application startup:
# bm25_payload = build_and_save_bm25()


In [ ]:
def load_bm25(cache_path=BM25_CACHE, validate=True):
  cache_path = Path(cache_path)
  if not cache_path.exists():
    raise FileNotFoundError(
        f"BM25 cache not found: {cache_path}. Run build_and_save_bm25() first."
    )

  with cache_path.open("rb") as f:
    # Load pickle file
    payload = pickle.load(f)

  if payload.get("format_version") != 1:
    raise ValueError("Unsupported BM25 cache format. Rebuild the index.")

  if validate:
    current = get_db_signature()
    cached = payload["signature"]
    if current != cached:
      raise RuntimeError(
                "BM25 cache may be stale. Database row count or max chunk_id changed. "
                "Run build_and_save_bm25() to refresh it."
            )

    print(
        f"Loaded BM25 index with {len(payload['chunk_ids']):,} chunks "
        f"(built {payload['built_at']})."
    )
    return payload

In [ ]:
def fetch_chunk_citations(chunk_ids):
  if not chunk_ids:
    # Return empty dataframe object
    return pd.DataFrame()

  # SQLite commonly limits bound parameters; batch IDs defensively.
  frames = []
  with connect_db() as conn:
    for start in range(0, len(chunk_ids), 900):
      batch = chunk_ids[start:start + 900]
      placeholders = ",".join("?" for _ in batch)
      sql_query = f"""
                SELECT chunk_id, doc_id, chunk_index, section,
                       page_start, page_end, text, filename,
                       source_url, download_url, source_title, source_type
                FROM chunk_citations
                WHERE chunk_id IN ({placeholders})
      """
      frames.append(pd.read_sql_query(sql_query, conn, params=batch))

  # Concatenate retrieved chunks to the result dataframe
  result = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
  order = {chunk_id: rank for rank, chunk_id in enumerate(chunk_ids)}

  # Re-arrange rows according to BM25 rank
  if not result.empty:
    # _rank_order is a temporary helper index column
    result["_rank_order"] = result["chunk_id"].map(order)
    # Sorts rows by ascending rank_order
    # .drop removes the column rank_order used for ranking
    result = result.sort_values("_rank_order").drop(columns="_rank_order")

  return result

In [ ]:
def bm25_search(query, bm25_payload, top_k=TOP_K):
  # Tokenise the query
  query_tokens = tokenise(query)
  scores = bm25_payload["bm25"].get_scores(query_tokens)
  k = min(top_k, len(scores))

  if k == 0:
    return pd.DataFrame()

  # argpartition avoids fully sorting large corpora; sort selected top-k
  candidate_idx = np.argpartition(scores, -k)[-k:]
  candidate_idx = candidate_idx[np.argsort(scores[candidate_idx])[::-1]]

  ranked_ids = [bm25_payload["chunk_ids"][i] for i in candidate_idx]
  ranked_scores = [float(scores[i]) for i in candidate_idx]

  results = fetch_chunk_citations(ranked_ids)
  score_map = dict(zip(ranked_ids, ranked_scores))
  results["bm25_score"] = results["chunk_id"].map(score_map)
  results["bm25_rank"] = results["chunk_id"].map(
      {chunk_id: rank for rank, chunk_id in enumerate(ranked_ids, start=1)}
  )
  return results.sort_values("bm25_rank").reset_index(drop=True)

In [ ]:
def load_embedding_matrix():
  """Loads embeddings as a matrix from SQLite database"""
  with connect_db() as conn:
    rows = conn.execute(
        """
        SELECT chunk_id, embedding
        FROM chunks
        WHERE embedding IS NOT NULL
        ORDER BY chunk_id
        """
    ).fetchall()

  ids, vectors = [], []
  # Create np array containing embeddings and matching chunk IDs
  for row in rows:
    vec = np.frombuffer(row["embedding"], dtype=EMBED_TYPE)
    ids.append(int(row["chunk_id"]))
    vectors.append(vec)

  if not vectors:
    raise ValueError("No embeddings found in chunks.embedding.")

  dims = {v.size for v in vectors}
  if len(dims) != 1:
    raise ValueError(f"Inconsistent embedding dimensions found: {sorted(dims)}")



  matrix = np.vstack(vectors)
  # Normalise rows: cosine similarity then equals dot product
  norms = np.linalg.norm(matrix, axis=1, keepdims=True)
  matrix = matrix / np.maximum(norms, 1e-12)
  return ids, matrix

vector_chunk_ids, vector_matrix = load_embedding_matrix()
print("Loaded vectors:", vector_matrix.shape)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Loaded vectors: (3719, 1024)


In [ ]:
def dense_search(query, top_k=TOP_K):
  """Performs dense search on vector database after encoding query."""
  model = SentenceTransformer(EMBEDDING_MODEL_NAME)
  query_vector = model.encode(
      [query],
      prompt_name="query",
      normalize_embeddings=True,
      convert_to_numpy=True
  )[0].astype(np.float32)

  assert vector_matrix.shape[1] == query_vector.shape[0], (
    f"Dimension mismatch: documents={vector_matrix.shape[1]}, "
    f"query={query_vector.shape[0]}"
  )

  # calculate similarity against all document vectors
  scores = vector_matrix @ query_vector

  # Take top_k chunks
  k = min(top_k, len(scores))

  # Order the np array
  idx = np.argpartition(scores, -k)[-k:]
  idx = idx[np.argsort(scores[idx])[::-1]]

  ranked_ids = [vector_chunk_ids[i] for i in idx]
  results = fetch_chunk_citations(ranked_ids)
  score_map = {vector_chunk_ids[i]: float(scores[i]) for i in idx}

  # Dense score refers to the score for a given chunk ID
  results["dense_score"] = results["chunk_id"].map(score_map)
  results["dense_rank"] = results["chunk_id"].map(
      {chunk_id: rank for rank, chunk_id in enumerate(ranked_ids, start=1)}
  )
  # Drop the sorting column after use
  return results.sort_values("dense_rank").reset_index(drop=True)

In [ ]:
def compare_search_algorithms(query, top_k=TOP_K):
  bm25_payload = load_bm25()

  # Perform lexical and semantic search
  lexical = bm25_search(query, bm25_payload, top_k=TOP_K)
  semantic = dense_search(query, top_k=TOP_K)

  # Compare semantic and lexical search along chunk_id
  # Merge all on chunk_id and filter later
  comparison = semantic[[
      "chunk_id", "doc_id", "filename", "source_title", "section",
      "page_start", "page_end", "text", "dense_score", "dense_rank"
  ]].merge(
      lexical[["chunk_id", "bm25_score", "bm25_rank"]],
      on="chunk_id",
      how="outer"
  )

  # Determine common chunks
  comparison["in_both"] = (
      comparison["dense_rank"].notna() &
      comparison["bm25_rank"].notna()
  )

  # Prepare values for display
  comparison = comparison.sort_values(
      ["in_both", "dense_rank", "bm25_rank"],
      ascending=[False, True, True],
      na_position="last"
  ).reset_index(drop=True)

  display(comparison)
  print(
      "Top-10 overlap:",
      len(set(semantic["chunk_id"]) & set(lexical["chunk_id"])),
      "chunks"
  )

In [ ]:
def main():
  compare_search_algorithms(QUERY="I am gay. Can I get married to a North Korean?")

In [ ]:
if __name__ == "__main__":
  main()

Loaded BM25 index with 3,719 chunks (built 2026-11-18T20:28:29.422367+00:00).


,chunk_id,doc_id,filename,source_title,section,page_start,page_end,text,dense_score,dense_rank,bm25_score,bm25_rank,in_both
0,255,14,Visa for Germany,Federal Foreign Office,I am not a German national and want to marry m...,10.0,10.0,First of all find out from the competent regis...,0.390436,5.0,21.247150,2.0,True
1,195,10,German Citizenship,Federal Foreign Office,I am married to a German national. Can I apply...,2.0,2.0,Provided certain other conditions are met you ...,0.371224,8.0,24.570504,1.0,True
2,96,7,Anti Sexual Discrimination,Federal Anti-Discrimination Agency,6. What is marriage for all?,2.0,2.0,"On 30 June 2017, the German Bundestag passed t...",0.404442,1.0,NaN,NaN,False
3,818,36,Internationale Studierende mit Kindern Aufenth...,Deutsches Studentenwerk,Beispiele,31.0,31.0,Die Studentin Nina aus Georgien ist schwanger....,0.404423,2.0,NaN,NaN,False
4,758,36,Internationale Studierende mit Kindern Aufenth...,Deutsches Studentenwerk,Ausnahmen von der Sprachanforderung:,14.0,15.0,"9 Der Nachzug darf nicht zugelassen werden, we...",0.397976,3.0,NaN,NaN,False
5,1128,39,Residency and Social Security Law,Deutsches Studentenwerk,Examples,29.0,30.0,The Chinese student Li is 22 years old and wis...,0.397240,4.0,NaN,NaN,False
6,759,36,Internationale Studierende mit Kindern Aufenth...,Deutsches Studentenwerk,HINWEIS,15.0,15.0,Bei einer erst während des Aufenthalts in Deut...,0.385576,6.0,NaN,NaN,False
7,2972,74,Information about the Taxation System in Germany,Uni Magdeburg,ATTENTION:,1.0,2.0,"your status as ""married"" will only be taken in...",0.373156,7.0,NaN,NaN,False
8,1176,39,Residency and Social Security Law,Deutsches Studentenwerk,1.9.4 Are there any other reasons for changing...,47.0,47.0,You can only obtain a residence permit other t...,0.363754,9.0,NaN,NaN,False
9,805,36,Internationale Studierende mit Kindern Aufenth...,Deutsches Studentenwerk,2.11.3 Tausch der Aufenthaltstitel zwischen de...,27.0,27.0,"Auch ein Tausch der ""Stammberechtigung"" zwisch...",0.362387,10.0,NaN,NaN,False


Top-10 overlap: 2 chunks
